## Fetch OHLCV data for Vietnamese stocks from *vnstock*

In [0]:
%pip install vnstock==3.5.1

In [0]:
%run ../config/config

In [0]:
import pandas as pd
import time
from datetime import datetime
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType,
    DoubleType, LongType, TimestampType,
)

vnstock_api_key = dbutils.secrets.get(scope=secret_scope, key=Vnstock_API_key_name)

#### 1. Parameters

In [0]:
exchanges = default_exchanges
start_date = data_start
end_date = data_end

print(f"Exchanges: {[ex.value for ex in exchanges]}, Period: {start_date} → {end_date}")

#### 2. Get stock list 

In [0]:
from vnstock import Vnstock, Listing, register_user

# Register API key from databricks secrets
register_user(api_key=vnstock_api_key)

# vnstock client
stock_client = Vnstock()

def get_stock_list(exchanges: list) -> list:
    """Get list of (symbol, Exchange) tuples for the given exchanges."""
    listing = Listing(source="VCI")
    all_symbols = listing.symbols_by_exchange()

    symbol_exchange_pairs = []
    for ex in exchanges:
        stocks = all_symbols[
            (all_symbols["exchange"] == ex.value) &
            (all_symbols["type"] == "STOCK")
        ]
        pairs = [(s, ex) for s in stocks["symbol"].tolist()]
        symbol_exchange_pairs.extend(pairs)
        print(f"Found {len(pairs)} symbols on {ex.value}")

    return symbol_exchange_pairs

symbol_exchange_pairs = get_stock_list(exchanges)
print(f"Total symbols across {[ex.value for ex in exchanges]}: {len(symbol_exchange_pairs)}")

####3. Fetch OHLCV 

In [0]:
def fetch_ohlcv(symbol: str, start: str, end: str, exchange: str, max_retries: int = 3) -> pd.DataFrame:
    """Fetch OHLCV data for a single symbol."""
    for attempt in range(max_retries):
        try:
            stock = stock_client.stock(symbol=symbol, source="VCI")
            df = stock.quote.history(start=start, end=end, interval="1D")

            if df is None or df.empty:
                print(f"{symbol}: No data returned")
                return pd.DataFrame()

            # Standardize column names 
            aliases = {
                "time": "date", "date": "date", "tradingdate": "date",
                "open": "open", "high": "high", "low": "low", "close": "close",
                "volume": "volume", "vol": "volume",
            }
            col_map = {col: aliases[col.lower().strip()]
                       for col in df.columns
                       if col.lower().strip() in aliases}
            df = df.rename(columns=col_map)

            required = ["date"] + ohlcv_columns
            missing = [c for c in required if c not in df.columns]
            if missing:
                print(f"{symbol}: Missing columns {missing}")
                return pd.DataFrame()

            df = df[required].copy()
            df["date"] = pd.to_datetime(df["date"])
            df["symbol"] = symbol
            df["exchange"] = exchange.value
            df["ingested_at"] = datetime.now()
            df["source"] = "vnstock"

            return df

        except Exception as e:
            err_str = str(e)
            if "502" in err_str or "429" in err_str or "Bad Gateway" in err_str or "Too Many" in err_str:
                wait = 10 * (attempt + 1)
                print(f"{symbol}: Attempt {attempt+1}/{max_retries} rate limited: {e}. waiting {wait}s")
                time.sleep(wait)
                continue
            print(f"{symbol}: Failed (no retry): {e}")
            return pd.DataFrame()

    print(f"{symbol}: All {max_retries} retries fail")
    return pd.DataFrame()

####4. Ingest all symbols

In [0]:
schema = StructType([
    StructField("symbol", StringType(), False),
    StructField("date", DateType(), False),
    StructField("open", DoubleType()),
    StructField("high", DoubleType()),
    StructField("low", DoubleType()),
    StructField("close", DoubleType()),
    StructField("volume", LongType()),
    StructField("exchange", StringType()),
    StructField("ingested_at", TimestampType()),
    StructField("source", StringType()),
])

write_batch_size = 100 
max_sympols_per_run = 200 
cooldown_between_batches = 30 

schema_columns = [f.name for f in schema.fields]

def flush_to_delta(dfs):
    """Write accumulated DataFrames to bronze Delta table."""
    if not dfs:
        return 0
    batch_pdf = pd.concat(dfs, ignore_index=True)[schema_columns]
    sdf = spark.createDataFrame(batch_pdf, schema=schema)
    sdf.write.mode("append").saveAsTable(tables["raw_ohlcv"])
    return len(batch_pdf)

# skip symbols exist
try:
    existing_df = spark.sql(f"SELECT DISTINCT symbol FROM {tables['raw_ohlcv']}").toPandas()
    already_ingested = set(existing_df["symbol"].tolist())
    print(f"{len(already_ingested)} symbols already in Delta (skip)")
except Exception as e:
    already_ingested = set()
    print(f"No existing data found (reason: {e})")

all_dfs = []
failed_symbols = []
last_call_time = 0.0
total = len(symbol_exchange_pairs)
total_rows_written = 0
symbols_fetched = 0
skipped_existing = 0

for i, (symbol, ex) in enumerate(symbol_exchange_pairs):
    if symbol in already_ingested:
        skipped_existing += 1
        continue

    # stop if hit the limit
    if symbols_fetched >= max_sympols_per_run:
        print(f"\nReached per-run limit ({max_sympols_per_run} symbols).")
        break

    # rate limit
    elapsed = time.time() - last_call_time
    if elapsed < API_call_interval:
        time.sleep(API_call_interval - elapsed)

    # progress reporting per batch
    if i % ingestion_batch_size == 0:
        print(f"Processing symbols {i+1}-{min(i+ingestion_batch_size, total)} of {total}")

    last_call_time = time.time()
    df = fetch_ohlcv(symbol, start_date, end_date, ex)
    if df.empty:
        failed_symbols.append((symbol, ex))
        continue

    all_dfs.append(df)
    symbols_fetched += 1

    
    if len(all_dfs) >= write_batch_size:
        rows = flush_to_delta(all_dfs)
        total_rows_written += rows
        print(f"  Flushed {rows:,} rows to Delta ({symbols_fetched} symbols so far)")
        all_dfs.clear()
        print(f"  Cool down {cooldown_between_batches}s")
        time.sleep(cooldown_between_batches)

rows = flush_to_delta(all_dfs)
total_rows_written += rows
all_dfs.clear()

remaining = total - skipped_existing - symbols_fetched - len(failed_symbols)
print(f"\nIngestion complete:")
print(f"  Symbols fetched this run: {symbols_fetched}")
print(f"  Total rows written: {total_rows_written:,}")
print(f"  Skipped (already in Delta): {skipped_existing}")
print(f"  Failed: {len(failed_symbols)}")
if remaining > 0:
    print(f"  Remaining: {remaining}")

####5. Verify ingestion

In [0]:
%sql
SELECT
    exchange,
    COUNT(DISTINCT symbol) AS num_symbols,
    COUNT(*)               AS total_rows,
    MIN(date)              AS min_date,
    MAX(date)              AS max_date
FROM wyckoff_catalog.bronze.raw_ohlcv
GROUP BY exchange
ORDER BY exchange;